# LangGraph "Prebuilt ToolNode & Conditional Edge" Pattern

In [1]:
# --- 0. 필요한 패키지 설치 (처음 한 번만 실행) ---
# 노트북 실행 전 필요한 패키지들을 설치합니다.
# 이미 설치되어 있다면 이 셀은 건너뛰어도 됩니다.

import sys
import subprocess
import os
from pathlib import Path
from dotenv import load_dotenv

# .env 파일 로드 (노트북에서는 여러 위치를 시도)
# 노트북은 notebook/ 폴더에 있으므로 상위 디렉토리(프로젝트 루트)에서 찾기
possible_env_paths = [
    Path.cwd() / ".env",  # 현재 작업 디렉토리
    Path.cwd().parent / ".env",  # 상위 디렉토리 (프로젝트 루트)
    Path(__file__).parent.parent / ".env" if '__file__' in globals() else None,  # 스크립트 실행 시
]

env_loaded = False
for env_path in possible_env_paths:
    if env_path and env_path.exists():
        load_dotenv(dotenv_path=env_path)
        print(f"✓ .env 파일 로드 완료: {env_path}")
        env_loaded = True
        break

if not env_loaded:
    print("⚠️ .env 파일을 찾을 수 없습니다. 환경 변수를 직접 설정하거나 .env 파일을 확인해주세요.")

# 패키지명과 임포트명 매핑
required_packages = {
    "langchain-google-genai>=2.0.0": "langchain_google_genai",
    "langchain-core>=0.3.0": "langchain_core",
    "langgraph>=0.2.0": "langgraph",
    "typing-extensions>=4.8.0": "typing_extensions",
    "python-dotenv>=1.0.0": "dotenv"
}

print("\n필요한 패키지 확인 중...\n")
for package_spec, import_name in required_packages.items():
    package_name = package_spec.split(">=")[0].split("==")[0]
    try:
        __import__(import_name)
        print(f"✓ {package_name} 이미 설치됨")
    except ImportError:
        print(f"✗ {package_name} 설치 필요 - 설치 중...")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", package_spec], 
                                stdout=subprocess.DEVNULL, stderr=subprocess.PIPE)
            print(f"✓ {package_name} 설치 완료")
        except subprocess.CalledProcessError as e:
            print(f"✗ {package_name} 설치 실패: {e}")
            raise

# 환경 변수 확인
print("\n환경 변수 확인:")
api_key = os.getenv("GEMINI_API_KEY")
model_name = os.getenv("GEMINI_MODEL_NAME", "gemini-3-flash-preview")

if api_key:
    print(f"✓ GEMINI_API_KEY 설정됨")
else:
    print(f"✗ GEMINI_API_KEY가 설정되지 않았습니다. .env 파일에 설정해주세요.")

print(f"✓ 모델: {model_name}")
print("\n모든 패키지 준비 완료!")


✓ .env 파일 로드 완료: c:\Users\user\Documents\Project\P_04_Scope\.env

필요한 패키지 확인 중...

✗ langchain-google-genai 설치 필요 - 설치 중...
✓ langchain-google-genai 설치 완료
✓ langchain-core 이미 설치됨
✓ langgraph 이미 설치됨
✓ typing-extensions 이미 설치됨
✓ python-dotenv 이미 설치됨

환경 변수 확인:
✓ GEMINI_API_KEY 설정됨
✓ 모델: gemini-3-flash-preview

모든 패키지 준비 완료!


In [2]:
from typing import Annotated, Literal
from typing_extensions import TypedDict
import os
from pathlib import Path
from dotenv import load_dotenv

# 환경 변수 로드 (이미 셀 0에서 로드했지만, 다시 확인)
# 노트북은 notebook/ 폴더에 있으므로 상위 디렉토리(프로젝트 루트)에서 찾기
possible_env_paths = [
    Path.cwd() / ".env",  # 현재 작업 디렉토리
    Path.cwd().parent / ".env",  # 상위 디렉토리 (프로젝트 루트)
]

for env_path in possible_env_paths:
    if env_path.exists():
        load_dotenv(dotenv_path=env_path, override=False)  # 이미 로드된 값은 덮어쓰지 않음
        break

# Google GenAI 모델 사용 (프로젝트 표준)
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage

# LangGraph 공식 구성요소 임포트
from langgraph.graph import StateGraph, START, END
from langgraph.graph import MessagesState  # 표준 메시지 관리 State
from langgraph.prebuilt import ToolNode, tools_condition  # 핵심: 미리 구현된 노드와 조건

# 모델 설정 (프로젝트 표준)
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
MODEL_NAME = os.getenv("GEMINI_MODEL_NAME", "gemini-3-flash-preview")

if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY가 설정되지 않았습니다. .env 파일에 설정해주세요.")

print(f"✓ Google GenAI 모델 초기화 준비 완료")
print(f"  모델: {MODEL_NAME}")

# --- 1. 도구(Tools) 정의 ---
# 기존의 Step 함수들을 @tool 데코레이터로 감싸서 정의합니다.
# Docstring은 LLM이 도구를 선택하는 기준이 되므로 매우 중요합니다.

@tool
def check_location_context(image_path: str):
    """
    [Step 1] 이미지의 위치적 맥락(접속점 여부, 전선 끝단 등)을 식별합니다.
    분석의 가장 첫 단계에서 실행하여 접속불량 분석이 유효한 위치인지 판단합니다.
    """
    # 실제로는 vision 모델 호출 로직이 들어감
    return {"location": "screw_terminal", "is_connection": True, "description": "나사 체결 단자"}

@tool
def analyze_color_spectrum(image_path: str):
    """
    [Step 2] 아산화동(Cu2O) 증식 여부를 판단하기 위해 붉은색/적갈색 패턴을 분석합니다.
    접속점이 확인된 후, 과열의 화학적 증거를 찾을 때 사용합니다.
    """
    return {"suspicious_color": True, "color_ratio": 0.35}

@tool
def analyze_thermal_gradient(image_path: str):
    """
    [Step 3] 전선의 탄화 패턴을 통해 열적 구배(Thermal Gradient)를 시각화합니다.
    열의 이동 방향과 발열원을 찾을 때 사용합니다.
    """
    return {"gradient_detected": True, "direction": "terminal_to_wire"}

@tool
def analyze_surface_texture(image_path: str):
    """
    [Step 4] 금속 표면의 곰보 자국(Pitting)이나 전기적 부식 흔적을 정밀 분석합니다.
    단락흔과 접촉불량 용융흔을 구별할 때 사용합니다.
    """
    return {"pitting_detected": True, "texture": "rough_erosion"}

# 사용할 도구 리스트
tools = [
    check_location_context, 
    analyze_color_spectrum, 
    analyze_thermal_gradient, 
    analyze_surface_texture
]

✓ Google GenAI 모델 초기화 준비 완료
  모델: gemini-3-flash-preview


In [3]:
# --- 2. 그래프 상태(State) 정의 ---
# MessagesState를 상속받으면 'messages' 필드가 자동으로 포함됩니다 (reducer 기능 포함)
class AgentState(MessagesState):
    # 필요하다면 추가 커스텀 필드 정의
    final_report: str = ""  # 기본값 설정

In [4]:
# --- 3. 에이전트 노드 정의 ---
def agent_node(state: AgentState):
    # 1. 모델 준비 (프로젝트 표준: Google GenAI Gemini)
    llm = ChatGoogleGenerativeAI(
        model=MODEL_NAME,
        google_api_key=GEMINI_API_KEY,
        temperature=0,
        convert_system_message_to_human=True  # Gemini는 시스템 메시지를 human 메시지로 변환 필요
    )
    
    # 2. 도구 바인딩 (LLM에게 도구 목록을 알려줌)
    llm_with_tools = llm.bind_tools(tools)
    
    # 3. 시스템 메시지 설정 (페르소나 부여)
    sys_msg = SystemMessage(content="""
        당신은 전기화재 감식 전문가 'Contact Expert'입니다.
        제공된 도구를 활용하여 이미지를 단계별로 분석하고, 
        접촉불량 여부를 판단하여 최종 리포트를 작성하세요.
        
        [절차]
        1. 먼저 위치를 확인(Step 1)하십시오.
        2. 필요한 경우 색상(Step 2), 열적 구배(Step 3), 표면(Step 4) 도구를 사용하십시오.
        3. 충분한 증거가 모이면 최종 결론을 내리십시오.
    """)
    
    # 4. 추론 실행
    # state["messages"]에는 이전 대화와 도구 실행 결과가 모두 들어있음
    response = llm_with_tools.invoke([sys_msg] + state["messages"])
    
    # 5. 결과 반환 (기존 메시지 리스트에 append 됨)
    return {"messages": [response]}

In [5]:
# --- 4. 그래프(Workflow) 구성 ---
def build_contact_expert_graph():
    # StateGraph 초기화
    workflow = StateGraph(AgentState)

    # 노드 추가
    workflow.add_node("agent", agent_node)
    
    # [중요] ToolNode 활용
    # 직접 도구 실행 로직을 짤 필요 없이, LangGraph가 제공하는 노드 사용
    workflow.add_node("tools", ToolNode(tools))

    # 엣지 연결
    workflow.add_edge(START, "agent")

    # [중요] 조건부 엣지 (Conditional Edge)
    # tools_condition: LLM이 도구를 호출했으면 "tools"로, 답변을 끝냈으면 END로 자동 분기
    workflow.add_conditional_edges(
        "agent",
        tools_condition,
    )

    # 도구 실행 후에는 다시 에이전트로 돌아와서 판단하도록 연결 (Loop)
    workflow.add_edge("tools", "agent")

    # 컴파일 (Runnable 생성)
    return workflow.compile()

In [6]:
# --- 5. 도구 테스트 (선택 사항) ---
# 각 도구가 제대로 작동하는지 확인
print("=== 도구 테스트 ===")
test_image_path = "/tmp/test_wire.jpg"

print("\n1. 위치 맥락 확인:")
result1 = check_location_context.invoke({"image_path": test_image_path})
print(f"   결과: {result1}")

print("\n2. 색상 스펙트럼 분석:")
result2 = analyze_color_spectrum.invoke({"image_path": test_image_path})
print(f"   결과: {result2}")

print("\n3. 열적 구배 분석:")
result3 = analyze_thermal_gradient.invoke({"image_path": test_image_path})
print(f"   결과: {result3}")

print("\n4. 표면 질감 분석:")
result4 = analyze_surface_texture.invoke({"image_path": test_image_path})
print(f"   결과: {result4}")


=== 도구 테스트 ===

1. 위치 맥락 확인:
   결과: {'location': 'screw_terminal', 'is_connection': True, 'description': '나사 체결 단자'}

2. 색상 스펙트럼 분석:
   결과: {'suspicious_color': True, 'color_ratio': 0.35}

3. 열적 구배 분석:
   결과: {'gradient_detected': True, 'direction': 'terminal_to_wire'}

4. 표면 질감 분석:
   결과: {'pitting_detected': True, 'texture': 'rough_erosion'}


In [7]:
# --- 6. 그래프 빌드 및 시각화 ---
app = build_contact_expert_graph()

# 그래프 구조 확인
print("=== 그래프 구조 ===")
print(app.get_graph())
print("\n그래프 시각화 (Mermaid 형식):")
try:
    mermaid_diagram = app.get_graph().draw_mermaid()
    print(mermaid_diagram)
except Exception as e:
    print(f"시각화 생성 실패: {e}")


=== 그래프 구조 ===
Graph(nodes={'__start__': Node(id='__start__', name='__start__', data=RunnableCallable(tags=None, recurse=True, explode_args=False, func_accepts={}), metadata=None), 'agent': Node(id='agent', name='agent', data=agent(tags=None, recurse=True, explode_args=False, func_accepts={}), metadata=None), 'tools': Node(id='tools', name='tools', data=tools(tags=None, recurse=True, explode_args=False, func_accepts={'config': ('N/A', <class 'inspect._empty'>), 'runtime': ('N/A', <class 'inspect._empty'>)}, _tools_by_name={'check_location_context': StructuredTool(name='check_location_context', description='[Step 1] 이미지의 위치적 맥락(접속점 여부, 전선 끝단 등)을 식별합니다.\n분석의 가장 첫 단계에서 실행하여 접속불량 분석이 유효한 위치인지 판단합니다.', args_schema=<class 'langchain_core.utils.pydantic.check_location_context'>, func=<function check_location_context at 0x00000190101640E0>), 'analyze_color_spectrum': StructuredTool(name='analyze_color_spectrum', description='[Step 2] 아산화동(Cu2O) 증식 여부를 판단하기 위해 붉은색/적갈색 패턴을 분석합니다.\n접속점이 확인된 후, 과열

In [8]:
# --- 7. 실제 테스트 실행 ---
from langchain_core.messages import HumanMessage

# 테스트 입력 준비
test_inputs = {
    "messages": [
        HumanMessage(content="이 전선 사진을 분석해서 접촉불량인지 확인해줘. 이미지 경로: /tmp/burnt_wire.jpg")
    ]
}

print("=== 그래프 실행 시작 ===\n")

# 스트리밍 실행 (중간 단계 확인용)
step_count = 0
for event in app.stream(test_inputs, stream_mode="values"):
    step_count += 1
    messages = event.get("messages", [])
    if messages:
        last_msg = messages[-1]
        msg_type = type(last_msg).__name__
        content = getattr(last_msg, "content", str(last_msg))
        
        print(f"[Step {step_count}] 노드: {event.get('__end__', '진행중')}")
        print(f"  메시지 타입: {msg_type}")
        
        # 도구 호출인 경우
        if hasattr(last_msg, "tool_calls") and last_msg.tool_calls:
            print(f"  도구 호출:")
            for tool_call in last_msg.tool_calls:
                print(f"    - {tool_call.get('name', 'unknown')}: {tool_call.get('args', {})}")
        # 일반 메시지인 경우
        elif content:
            # 내용이 너무 길면 일부만 표시
            content_preview = content[:200] + "..." if len(content) > 200 else content
            print(f"  내용: {content_preview}")
        
        print()


=== 그래프 실행 시작 ===

[Step 1] 노드: 진행중
  메시지 타입: HumanMessage
  내용: 이 전선 사진을 분석해서 접촉불량인지 확인해줘. 이미지 경로: /tmp/burnt_wire.jpg

[Step 2] 노드: 진행중
  메시지 타입: AIMessage
  도구 호출:
    - check_location_context: {'image_path': '/tmp/burnt_wire.jpg'}

[Step 3] 노드: 진행중
  메시지 타입: ToolMessage
  내용: {"location": "screw_terminal", "is_connection": true, "description": "나사 체결 단자"}

[Step 4] 노드: 진행중
  메시지 타입: AIMessage
  도구 호출:
    - analyze_color_spectrum: {'image_path': '/tmp/burnt_wire.jpg'}

[Step 5] 노드: 진행중
  메시지 타입: ToolMessage
  내용: {"suspicious_color": true, "color_ratio": 0.35}

[Step 6] 노드: 진행중
  메시지 타입: AIMessage
  도구 호출:
    - analyze_thermal_gradient: {'image_path': '/tmp/burnt_wire.jpg'}

[Step 7] 노드: 진행중
  메시지 타입: ToolMessage
  내용: {"gradient_detected": true, "direction": "terminal_to_wire"}

[Step 8] 노드: 진행중
  메시지 타입: AIMessage
  도구 호출:
    - analyze_surface_texture: {'image_path': '/tmp/burnt_wire.jpg'}

[Step 9] 노드: 진행중
  메시지 타입: ToolMessage
  내용: {"pitting_detected": true, "texture": "roug

In [12]:
# --- 8. 최종 결과 확인 ---
# 마지막 상태에서 최종 메시지 확인
final_state = app.invoke(test_inputs)
final_messages = final_state.get("messages", [])

print("=== 최종 결과 ===")
print(f"총 메시지 수: {len(final_messages)}\n")

for i, msg in enumerate(final_messages, 1):
    msg_type = type(msg).__name__
    print(f"[{i}] {msg_type}")
    
    if hasattr(msg, "content") and msg.content:
        content_preview = msg.content[:300] + "..." if len(msg.content) > 300 else msg.content
        print(f"    내용: {content_preview}")
    
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        print(f"    도구 호출: {len(msg.tool_calls)}개")
        for tc in msg.tool_calls:
            print(f"      - {tc.get('name', 'unknown')}")
    
    print()


=== 최종 결과 ===
총 메시지 수: 10

[1] HumanMessage
    내용: 이 전선 사진을 분석해서 접촉불량인지 확인해줘. 이미지 경로: /tmp/burnt_wire.jpg

[2] AIMessage
    도구 호출: 1개
      - check_location_context

[3] ToolMessage
    내용: {"location": "screw_terminal", "is_connection": true, "description": "나사 체결 단자"}

[4] AIMessage
    도구 호출: 1개
      - analyze_color_spectrum

[5] ToolMessage
    내용: {"suspicious_color": true, "color_ratio": 0.35}

[6] AIMessage
    도구 호출: 1개
      - analyze_thermal_gradient

[7] ToolMessage
    내용: {"gradient_detected": true, "direction": "terminal_to_wire"}

[8] AIMessage
    도구 호출: 1개
      - analyze_surface_texture

[9] ToolMessage
    내용: {"pitting_detected": true, "texture": "rough_erosion"}

[10] AIMessage
    내용: [{'type': 'text', 'text': "전기화재 감식 전문가 'Contact Expert'로서 /tmp/burnt_wire.jpg 이미지를 정밀 분석한 결과, 해당 전선의 소손은 **접촉불량(Loose Connection)에 의한 과열**로 판단됩니다.\n\n### [감식 분석 결과 리포트]\n\n**1. 위치 분석 (Location Context)**\n- 분석 결과, 소손 부위가 나사 체결 방식의 **단자대(Screw Terminal)** 접속점으로 확인되었습니다. 이는 전기적 저항이 